# Multimodal Emotion Recognition — Exploration Notebook

Group 14 · EEG + Facial Analysis

This notebook covers:
1. EEG signal visualisation & band-power analysis
2. Facial dataset class distribution
3. Quick training run (smoke test)
4. Confusion matrix & per-class breakdown
5. Grad-CAM + EEG attention visualisation

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import torch

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. EEG Signal Visualisation

In [ ]:
from src.data.dataset import simulate_data
from src.data.eeg_pipeline import bandpass_filter, FREQ_BANDS

# Simulate a single EEG segment
X, y = simulate_data(n_samples=1, seq_len=512, n_features=1)
raw_signal = X[0, :, 0]   # (512,)

fs = 128.0
filtered = bandpass_filter(raw_signal, lowcut=0.5, highcut=45.0, fs=fs)

time = np.arange(len(raw_signal)) / fs

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(time, raw_signal, color='#94A3B8', linewidth=0.8)
axes[0].set_title('Raw EEG Signal')
axes[0].set_ylabel('Amplitude')

axes[1].plot(time, filtered, color='#2563EB', linewidth=0.8)
axes[1].set_title('Band-pass Filtered (0.5–45 Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## 2. Frequency-Band Power Analysis

In [ ]:
from scipy.signal import welch

freqs, psd = welch(filtered, fs=fs, nperseg=256)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs, psd, color='#0F172A', linewidth=1.2)

band_colors = {'delta':'#BFDBFE','theta':'#A7F3D0','alpha':'#FDE68A',
               'beta':'#FECACA','gamma':'#DDD6FE'}
for band, (lo, hi) in FREQ_BANDS.items():
    ax.axvspan(lo, hi, alpha=0.3, color=band_colors[band], label=band)

ax.set_xlim(0, 50)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power Spectral Density')
ax.set_title('EEG Power Spectrum with Frequency Bands')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Model Architecture Summary

In [ ]:
from src.models.multimodal_model import MultimodalEmotionModel

model = MultimodalEmotionModel()
counts = model.parameter_count()

print(f"Total parameters  : {counts['total']:,}")
print(f"Trainable params  : {counts['trainable']:,}")
print()

# Component breakdown
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {n:>10,} params")

## 4. Quick Training Run (Simulated Data)

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from src.training.trainer import Trainer, build_scheduler, resolve_device

device = resolve_device('auto')

# Simulated data
X, y = simulate_data(n_samples=500, seq_len=128, n_features=32)
face_X = torch.randn(500, 1, 48, 48)
eeg_T, y_T = torch.tensor(X), torch.tensor(y)

ds = TensorDataset(eeg_T, face_X, y_T)
n_tr = int(0.7 * len(ds))
n_vl = int(0.15 * len(ds))
n_te = len(ds) - n_tr - n_vl
tr_ds, vl_ds, te_ds = random_split(ds, [n_tr, n_vl, n_te])

kw = dict(batch_size=32, num_workers=0)
tr_ldr = DataLoader(tr_ds, shuffle=True, **kw)
vl_ldr = DataLoader(vl_ds, **kw)
te_ldr = DataLoader(te_ds, **kw)

model = MultimodalEmotionModel().to(device)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sch   = build_scheduler(opt, 'cosine', 5)
crit  = nn.CrossEntropyLoss()

trainer = Trainer(model, opt, crit, device, sch,
                  log_dir='../logs/nb', ckpt_dir='../checkpoints',
                  patience=5, use_amp=False)

def fwd(batch):
    eeg, face, labels = batch
    return model(face.to(device), eeg.to(device))['logits']

history = trainer.fit(tr_ldr, vl_ldr, epochs=5,
                      forward_fn=fwd, model_name='nb_test')
print('Done!')

## 5. Training Curves

In [ ]:
from src.evaluation.metrics import plot_training_curves
plot_training_curves(history, filename='nb_curves.png')
from IPython.display import Image
Image('../results/nb_curves.png')

## 6. Confusion Matrix

In [ ]:
from src.evaluation.metrics import plot_confusion_matrix, error_analysis

model.eval()
all_p, all_l = [], []
with torch.no_grad():
    for batch in te_ldr:
        eeg, face, lbl = batch
        logits = model(face.to(device), eeg.to(device))['logits']
        all_p.extend(logits.argmax(-1).cpu().tolist())
        all_l.extend(lbl.tolist())

all_l, all_p = np.array(all_l), np.array(all_p)
plot_confusion_matrix(all_l, all_p, filename='nb_cm.png')
error_analysis(all_l, all_p)
Image('../results/nb_cm.png')

## 7. EEG Temporal Attention Visualisation

In [ ]:
from src.evaluation.explainability import plot_eeg_attention

# Get a single sample
eeg_sample  = eeg_T[0:1].to(device)
face_sample = face_X[0:1].to(device)

model.eval()
with torch.no_grad():
    out = model(face_sample, eeg_sample)

attn_weights = out['eeg_attn'][0].cpu().numpy()  # (T,)
predicted_cls = out['logits'][0].argmax().item()
label_names = ['anger', 'fear', 'sadness', 'disgust']

plot_eeg_attention(attn_weights,
                   emotion_label=label_names[predicted_cls],
                   save_path='../results/nb_eeg_attn.png')
Image('../results/nb_eeg_attn.png')